# Описание полученного варианта

## Вариант 3 (№21 в списке): 
- Исследуемые поля:
    - timestamp
    - hour
    - is_weekend
    - device_type
    - os_family
    - browser_family
    - known_device
- Основная проверка:
Временное разделение по timestamp

In [1]:
# Подготовка проекта

from pathlib import Path
import sys
import numpy as np
import pandas as pd
import sklearn

RANDOM_STATE = 42
DATA_PATH = Path('data/auth_events_lab01.xlsx')

print(sys.version)
print('pandas:', pd.__version__)
print('scikit-learn:', sklearn.__version__)

3.11.16 (main, Aug 13 2026, 09:46:36) [GCC 15.2.0]
pandas: 2.2.2
scikit-learn: 1.4.2


## Постановка задачи и паспорт датасета

### Прикладная задача ИБ

Требуется оценить, является ли конкретная попытка аутентификации подозрительной, используя контекст устройства и времени события (тип устройства, семейство ОС, семейство браузера, признак известного устройства, час суток, признак выходного дня). Результат предназначен для аналитика SOC / системы мониторинга ИБ: он позволяет приоритизировать проверки, повысить уровень логирования для подозрительных сессий, запросить дополнительный фактор аутентификации, ограничить доступ к чувствительным ресурсам или инициировать ручную верификацию события. Автоматическое блокирование учетной записи по одной метке не предполагается — решение носит характер поддержки принятия решений.

### Объект анализа и гранулярность наблюдения

**Объект анализа** — одна попытка аутентификации пользователя.

**Гранулярность наблюдения** — одна строка таблицы соответствует ровно одной попытке входа, зафиксированной средством защиты. 

Одна строка не соответствует одному пользователю или инциденту, так как:

- один пользователь за период наблюдения совершает много попыток, поэтому строка на пользователя потеряла бы событийный контекст;
- один инцидент может объединять несколько попыток (например, серию неуспешных входов с последующим успехом), поэтому строка на инцидент смешала бы разные моменты принятия решения;
- решение принимается **в момент попытки**, поэтому единица наблюдения — именно попытка.

### Целевая переменная `is_suspicious`

**Целевая переменная:** `is_suspicious` — бинарная метка, отражающая итоговую оценку события.

- `1` — событие признано **подозрительным** (повышенный риск);
- `0` — событие признано **нормальным** (риск в пределах допустимого).

Имя переменной `is_suspicious` не является аргументом в пользу корректности разметки. Оно задает только семантику целевого признака — заявленный смысл "подозрительное / нормальное событие", — но не подтверждает, что проставленные в наборе метки действительно соответствуют реальному классу объекта.

Корректность разметки должна обосновываться отдельно и независимо от названия столбца: через источник меток (кто или что их формирует) и через процедуру их получения (по какому правилу и в какой момент). Пока источник и процедура не проверены, имя `is_suspicious` остается лишь гипотезой о смысле признака, а не доказательством качества метки.

### Паспорт датасета

**Источник:** лист «Данные», из таблицы, прикрепленной к лабораторной работе (https://disk.yandex.ru/i/jROt0cPOXj_qHA).

**Дата получения:** 23.09.2026.

**Исходный размер:** 163 Kb

**Ограничения:** 

Набор синтетически сгенерированный, поэтому существует ряд ограничений:

- распределения признаков и метки могут не соответствовать реальному SOC;
- источник не содержит реальных персональных, конфиденциальных или опасных данных;
- правила генерации метки и признаков неизвестны, поэтому возможно наличие скрытых зависимостей между полями;
- часть полей (`analyst_verdict`, `analyst_risk_score`) появляется **после** события и не должна использоваться при принятии решения.

### Таблица признаков

| Столбец | Смысл | Фактический тип | Ожидаемый тип | Допустимые значения | Момент доступности | Решение |
|---|---|---|---|---|---|---|
| `event_id` | Технический идентификатор записи | string | string | уникальная строка | до события (служебный) | исключить |
| `timestamp` | Время попытки аутентификации | datetime | datetime | в пределах периода сбора | в момент события | использовать только для разделения |
| `user_id` | Псевдоним учетной записи | category | category | `usr_XXX` | в момент события | использовать как группу при анализе, из X исключить |
| `department` | Подразделение пользователя | category | category | IT, HR, SOC, R&D, Sales, Finance, Operations | в момент события | оставить как контекст |
| `source_ip` | IP-адрес источника | string | string | IPv4 | в момент события | оставить как контекст |
| `country` | Страна по IP | category | category | ISO-код | в момент события | оставить как контекст |
| `device_type` | Тип устройства | category | category | desktop, laptop, tablet, mobile | в момент события | оставить |
| `os_family` | Семейство ОС | category | category | Windows, macOS, Linux, iOS, Android | в момент события | оставить |
| `browser_family` | Семейство браузера | category | category | Chrome, Firefox, Safari, Edge | в момент события | оставить |
| `auth_method` | Способ аутентификации | category | category | password, otp, push, fido2 | в момент события | оставить как контекст |
| `hour` | Час суток по UTC | integer | integer | 0...23 | в момент события | оставить |
| `is_weekend` | Признак выходного дня | binary | binary | {0, 1} | в момент события | оставить |
| `failed_attempts_24h` | Неуспешных попыток за 24 ч | integer | integer | >= 0 | в момент события | оставить как контекст |
| `login_velocity_1h` | Попыток входа за последний час | integer | integer | >= 0 | в момент события | оставить как контекст |
| `distance_from_usual_km` | Расстояние от обычной геолокации | float | float | 0...20000 | в момент события | оставить как контекст |
| `account_age_days` | Возраст учетной записи, дни | integer | integer | >= 0 | в момент события | оставить как контекст |
| `password_age_days` | Возраст текущего пароля, дни | string | integer | >= 0 | в момент события | оставить как контекст |
| `known_device` | Известно ли устройство системе | binary | binary | {0, 1} | в момент события | оставить |
| `new_country` | Новая ли страна для пользователя | binary | binary | {0, 1} | в момент события | оставить как контекст |
| `vpn_used` | Обнаружено ли использование VPN | binary | binary | {0, 1} | в момент события | оставить как контекст |
| `analyst_verdict` | Вердикт аналитика | category | category | benign, confirmed_threat | после решения | исключить (утечка) |
| `analyst_risk_score` | Оценка риска аналитиком | integer | integer | 0...100 | после решения | исключить (утечка) |
| `is_suspicious` | Целевая метка | binary | binary | {0, 1} | после события | это `y`, в `X` не входит |

- **`y`** — целевая переменная `is_suspicious`.
- **`X`** — матрица признаков: все столбцы, кроме `y` и полей, которые исключены по причине утечки.

In [2]:
# Загрузка и первичная инвентаризация

pd.set_option('display.max_columns', None)
df = pd.read_excel(DATA_PATH, sheet_name='data')
df_raw = df.copy()

original_shape = df.shape
original_rows, original_cols = original_shape

print(f'Загружен файл: {DATA_PATH}')

print('=== Первые 5 строк ===')
display(df.head(5))

print('\n=== Последние 5 строк ===')
display(df.tail(5))

original_rows, original_cols = df.shape
print('=== Исходный размер таблицы ===')
print(f'Число строк:   {original_rows}')
print(f'Число столбцов: {original_cols}')

print('\n=== Названия столбцов и типы данных ===')
print(df.dtypes)

mem = df.memory_usage(deep=True)
print(f'\nОбъем памяти: {mem.sum() / 1024:.3f} Кб')

# Целевой столбец присутствует ровно один раз
target_count = (df.columns == 'is_suspicious').sum()
print(f'\nСтолбцов с именем is_suspicious: {target_count}')
assert target_count == 1, 'Ожидался один столбец is_suspicious'

dict_df = pd.read_excel(DATA_PATH, sheet_name='Словарь')

# Поиск ожидаемых типов в соответствии со словарем
field_col = next(c for c in dict_df.columns if 'Поле' in c)
type_col  = next(c for c in dict_df.columns if 'Ожидаемый тип' in c)

# Словарь {поле: ожидаемый_тип}
expected_fields = (
    dict_df[[field_col, type_col]]
    .set_index(field_col)[type_col]
    .to_dict()
)

# Числовые - integer / float
numeric_cols = [col for col, t in expected_fields.items() if t in ('integer', 'float')]

print('\n=== Описательные статистики числовых признаков ===')
display(df[numeric_cols].describe().T)

# Категориальные - category
cat_cols = [col for col, t in expected_fields.items() if t == 'category' and col in df.columns]

summary = []
for col in cat_cols:
    vc = df[col].value_counts(dropna=True)
    top3 = vc.head(3)
    summary.append({
        'Столбец': col,
        'Уникальных': int(df[col].nunique(dropna=True)),
        'Топ-1': f'{top3.index[0]!r} ({top3.iloc[0]})' if len(top3) > 0 else '-',
        'Топ-2': f'{top3.index[1]!r} ({top3.iloc[1]})' if len(top3) > 1 else '-',
        'Топ-3': f'{top3.index[2]!r} ({top3.iloc[2]})' if len(top3) > 2 else '-',
    })

print('\n=== Описание категориальных признаков ===')
display(pd.DataFrame(summary))

Загружен файл: data/auth_events_lab01.xlsx
=== Первые 5 строк ===


,event_id,timestamp,user_id,department,source_ip,country,device_type,os_family,browser_family,auth_method,hour,is_weekend,failed_attempts_24h,login_velocity_1h,distance_from_usual_km,account_age_days,password_age_days,known_device,new_country,vpn_used,analyst_verdict,analyst_risk_score,is_suspicious
0,evt_001370,2026-03-12 20:27:57,usr_046,Operations,203.0.113.178,SE,mobile,Android,Chrome,password,20,0,0,1,39.4,871,229,0,0,0,benign,65,0
1,evt_000493,2026-01-06 00:17:20,usr_068,Sales,203.0.113.132,RU,mobile,Android,Chrome,password,0,0,0,5,16.9,2383,11,1,0,0,benign,43,0
2,evt_000433,2026-01-27 17:57:11,usr_033,SOC,192.0.2.185,PL,laptop,Windows,Chrome,password,17,0,2,5,28.8,795,258,1,0,0,benign,50,0
3,evt_001361,2026-03-10 16:30:25,usr_085,Sales,192.0.2.91,NO,mobile,Android,Chrome,otp,16,0,0,3,26.1,1186,184,1,0,0,benign,40,0
4,evt_000791,2026-02-10 05:26:56,usr_092,R&D,203.0.113.190,DE,laptop,Windows,Chrome,fido2,5,0,3,6,86.0,784,182,1,0,0,benign,7,0



=== Последние 5 строк ===


,event_id,timestamp,user_id,department,source_ip,country,device_type,os_family,browser_family,auth_method,hour,is_weekend,failed_attempts_24h,login_velocity_1h,distance_from_usual_km,account_age_days,password_age_days,known_device,new_country,vpn_used,analyst_verdict,analyst_risk_score,is_suspicious
1425,evt_000907,2026-03-09 00:01:20,usr_065,Operations,203.0.113.93,PL,tablet,iOS,Safari,push,0,0,0,2,49.8,3245,79,1,0,0,benign,15,0
1426,evt_001094,2026-01-17 09:15:30,usr_051,HR,192.0.2.165,RU,tablet,iOS,Safari,password,9,1,0,2,64.2,1803,239,1,0,0,benign,46,0
1427,evt_001253,2026-01-25 00:38:56,usr_061,R&D,198.51.100.128,RU,desktop,Windows,Chrome,otp,0,1,5,13,1088.3,2917,177,1,1,0,confirmed_threat,78,1
1428,evt_000743,2026-02-23 22:19:53,usr_043,HR,192.0.2.51,US,laptop,Windows,Edge,otp,22,0,1,5,14.4,1495,257,1,0,0,benign,31,0
1429,evt_000581,2026-02-14 19:29:43,usr_019,Finance,198.51.100.3,NL,laptop,Windows,Chrome,password,19,1,0,2,73.4,2915,unknown,1,0,0,benign,24,0


=== Исходный размер таблицы ===
Число строк:   1430
Число столбцов: 23

=== Названия столбцов и типы данных ===
event_id                   object
timestamp                  object
user_id                    object
department                 object
source_ip                  object
country                    object
device_type                object
os_family                  object
browser_family             object
auth_method                object
hour                        int64
is_weekend                  int64
failed_attempts_24h         int64
login_velocity_1h           int64
distance_from_usual_km    float64
account_age_days            int64
password_age_days          object
known_device                int64
new_country                 int64
vpn_used                    int64
analyst_verdict            object
analyst_risk_score          int64
is_suspicious               int64
dtype: object

Объем памяти: 1138.146 Кб

Столбцов с именем is_suspicious: 1

=== Описательные статистики 

,count,mean,std,min,25%,50%,75%,max
hour,1430.0,11.731469,7.113637,0.0,6.00,12.00,18.000,23.0
failed_attempts_24h,1430.0,1.897902,37.336751,-1.0,0.00,0.00,1.000,999.0
login_velocity_1h,1430.0,3.197902,2.222254,1.0,2.00,3.00,4.000,18.0
distance_from_usual_km,1398.0,376.536409,1853.187377,-25.0,15.70,31.35,56.275,20000.0
account_age_days,1430.0,1637.660140,921.062124,-7.0,839.75,1656.50,2365.500,3266.0
analyst_risk_score,1430.0,40.848951,23.968772,3.0,20.00,40.00,59.000,99.0



=== Описание категориальных признаков ===


,Столбец,Уникальных,Топ-1,Топ-2,Топ-3
0,user_id,120,'usr_062' (24),'usr_025' (21),'usr_082' (19)
1,department,8,'SOC' (269),'R&D' (248),'HR' (196)
2,country,9,'NO' (393),'DE' (268),'SE' (257)
3,device_type,4,'mobile' (491),'tablet' (334),'laptop' (329)
4,os_family,5,'Android' (464),'Windows' (409),'iOS' (361)
5,browser_family,5,'Chrome' (793),'Safari' (263),'Edge' (182)
6,auth_method,4,'password' (571),'push' (372),'otp' (330)
7,analyst_verdict,2,'benign' (1301),'confirmed_threat' (129),-


## Числовые признаки

Как можно заметить из таблицы описательных статистик числовых признаков, признак `password_age_days` отсутствует. Это связано с тем, что помимо чисел в изначальном датасете содержатся ячейки со значением `unknown`, из-за чего pandas определяет тип данного признака как `object` (т.е. как строку). Исходя из этого, была найдена причина того, что признак не выводится в таблице — метод `describe` может работать только с числовыми типами.

In [3]:
# Проверка типов и допустимых значений

type_report = pd.DataFrame({
    'фактический_тип': df.dtypes,
    'ожидаемый_тип': pd.Series(expected_fields)
})
display(type_report)

,фактический_тип,ожидаемый_тип
event_id,object,string
timestamp,object,datetime
user_id,object,category
department,object,category
source_ip,object,string
country,object,category
device_type,object,category
os_family,object,category
browser_family,object,category
auth_method,object,category


## Предметные ограничения

- `timestamp` — привести к `datetime`, посчитать строки, которые не соответствуют формату;
- `hour` — целое в диапазоне 0...23, согласованное с `timestamp`;
- `is_weekend` — {0, 1}, согласованное с `timestamp`;
- бинарные `known_device`, `new_country`, `vpn_used`, `is_suspicious` — строго {0, 1};
- числовые счетчики, `account_age_days` — целое неотрицательное число;
- `distance_from_usual_km` — дробное неотрицательное число;
- `password_age_days` — привести к числу, а также не больше `account_age_days`;
- категориальные `device_type`, `os_family`, `browser_family`, `department` — не пустые значения;
- `event_id`, `user_id`, `source_ip` — в формате `evt_xxxxxx`, `usr_xxx` и `IPv4` соответственно (`x` — цифра).

In [4]:
# Валидация event_id

def is_valid_event_id(v):
    # Не строка - невалидно
    if not isinstance(v, str):
        return False
    # Должно быть ровно 2 части при split('_')
    parts = v.split('_')
    if len(parts) != 2:
        return False
    prefix, digits = parts
    # Первая часть - 'evt'
    if prefix != 'evt':
        return False
    # Вторая часть - число из 6 цифр
    return digits.isdigit() and len(digits) == 6


event_id_valid = df['event_id'].map(is_valid_event_id)
event_invalid = int((~event_id_valid).sum())
print(f'event_id не соответствуют формату: {event_invalid}')
display(df.loc[~event_id_valid, ['event_id']])

# Валидация user_id

def is_valid_user_id(v):
    # Не строка - невалидно
    if not isinstance(v, str):
        return False
    # Должно быть ровно 2 части при split('_')
    parts = v.split('_')
    if len(parts) != 2:
        return False
    prefix, digits = parts
    # Первая часть - 'usr'
    if prefix != 'usr':
        return False
    # Вторая часть - число из 3 цифр
    return digits.isdigit() and len(digits) == 3


user_id_valid = df['user_id'].map(is_valid_user_id)
user_invalid = int((~user_id_valid).sum())
print(f'user_id не соответствуют формату: {user_invalid}')
display(df.loc[~user_id_valid, ['user_id']])

# Валидация event_id

def is_valid_source_ip(v):
    # Не строка - невалидно
    if not isinstance(v, str):
        return False
    # Должно быть ровно 4 части при split('.')
    parts = v.split('.')
    if len(parts) != 4:
        return False
    # 4 октета, каждый из которых число <=255
    for octet in parts:
        if not octet.isdigit(): return False
        if int(octet) > 255 or int(octet) < 0: return False

    return True


source_ip_valid = df['source_ip'].map(is_valid_source_ip)
ip_invalid = int((~source_ip_valid).sum())
print(f'source_ip не соответствуют формату: {ip_invalid}')
display(df.loc[~source_ip_valid, ['source_ip']])

# Сколько строк не удалось распарсить как дату

ts_raw = df['timestamp']
# Замена невалидных на NaT (not a timestamp)
ts_parsed = pd.to_datetime(ts_raw, errors='coerce')

n_ts_invalid = int(ts_parsed.isna().sum() - ts_raw.isna().sum())

print(f'Всего строк: {len(df)}')
print(f'Пропусков timestamp: {int(ts_raw.isna().sum())}')
print(f'Не распарсилось в datetime: {n_ts_invalid}')

if n_ts_invalid:
    bad_rows = ts_raw[ts_parsed.isna() & ts_raw.notna()]
    print('\nПримеры непарсящихся значений:')
    display(bad_rows.head(10).to_frame('timestamp_raw'))

# Подстановка пустых значений вместо невалидных для последуюбщей обработки
df['timestamp'] = ts_parsed

# hour должен быть 0...23
hour = pd.to_numeric(df['hour'], errors='coerce')
n_hour_out_of_range = int(((hour < 0) | (hour > 23)).sum())
print(f'hour вне диапазона 0...23: {n_hour_out_of_range}')
if n_hour_out_of_range: display(df.loc[mask_hour_bad, ['timestamp', 'hour']])

# hour должен совпадать с часом из timestamp
mask_ts_ok = df['timestamp'].notna()
mismatch_hour = (df.loc[mask_ts_ok, 'hour'] != df.loc[mask_ts_ok, 'timestamp'].dt.hour)
n_hour_mismatch = int(mismatch_hour.sum())
print(f'hour не совпадает с timestamp.dt.hour: {n_hour_mismatch}')
if n_hour_mismatch:
    cols = ['timestamp', 'hour']
    display(
        df.loc[mask_ts_ok]
          .loc[mismatch_hour.values, cols]
          .assign(expected_hour=lambda d: d['timestamp'].dt.hour)
    )

# is_weekend должен совпадать с днем недели из timestamp
expected_weekend = (df.loc[mask_ts_ok, 'timestamp'].dt.dayofweek >= 5).astype(int)
mismatch_weekend = (df.loc[mask_ts_ok, 'is_weekend'] != expected_weekend)
n_weekend_mismatch = int(mismatch_weekend.sum())
print(f'is_weekend не совпадает с timestamp: {n_weekend_mismatch}')
if n_weekend_mismatch:
    cols = ['timestamp', 'is_weekend']
    display(
        df.loc[mask_ts_ok]
          .loc[mismatch_weekend.values, cols]
          .assign(expected_is_weekend=lambda d: (d['timestamp'].dt.dayofweek >= 5).astype(int))
    )

binary_cols = ['is_weekend', 'known_device', 'new_country', 'vpn_used', 'is_suspicious']

# Соответствие бинарных значений
binary_report = []
for col in binary_cols:
    vals = pd.to_numeric(df[col], errors='coerce')
    bad = ~vals.isin([0, 1]) & vals.notna()
    binary_report.append({
        'столбец': col,
        'dtype': str(df[col].dtype),
        'вне {0,1}': int(bad.sum()),
        'пропусков': int(df[col].isna().sum())
    })

print('\n=== Статистика по бинарным признакам ===')
display(pd.DataFrame(binary_report))

nonneg_cols = [
    'failed_attempts_24h', 'login_velocity_1h',
    'distance_from_usual_km', 'account_age_days',
    'password_age_days', 'analyst_risk_score',
]

# Соответствие счетчиков и числовых значений заданным диапазонам
nonneg_report = []
for col in nonneg_cols:
    vals = pd.to_numeric(df[col], errors='coerce')
    negative = (vals < 0) & vals.notna()
    not_numeric = vals.isna() & df[col].notna()
    nonneg_report.append({
        'столбец': col,
        'dtype': str(df[col].dtype),
        'отрицательных': int(negative.sum()),
        'нечисловых': int(not_numeric.sum()),
        'min': vals.min(),
        'max': vals.max(),
    })

print('\n=== Статистика по числовым признакам ===')
display(pd.DataFrame(nonneg_report))

# Отдельная проверка для password_age_days
col_pwd = 'password_age_days'
col_acc = 'account_age_days'

# Замена unknown на NaN для сравнения
pwd_numeric = pd.to_numeric(df[col_pwd], errors='coerce')
df[col_pwd] = pwd_numeric

valid = df[[col_pwd, col_acc]].notna().all(axis=1)
mask_pwd_gt_acc = valid & (df[col_pwd] > df[col_acc])

n_pwd_gt_acc = int(mask_pwd_gt_acc.sum())
print(f'Строк, где password_age_days > account_age_days: {n_pwd_gt_acc}')

if n_pwd_gt_acc:
    display(
        df.loc[mask_pwd_gt_acc,
               ['event_id', col_pwd, col_acc]]
          .head(20)
    )

cat_focus = [
    'device_type', 'os_family', 'browser_family',
    'country', 'department', 'auth_method', 'analyst_verdict',
]

print('\n=== Статистика по категориальным признакам ===')
for col in cat_focus:
    if col not in df.columns:
        continue
    vc = df[col].value_counts(dropna=False)
    print(f'\n=== {col} (уникальных: {df[col].nunique(dropna=True)}, '
          f'пропусков: {int(df[col].isna().sum())}) ===')
    display(vc.to_frame('частота'))

event_id не соответствуют формату: 12


,event_id
32,evt_retry_010
167,evt_retry_004
253,evt_retry_003
391,evt_retry_008
444,evt_retry_007
603,evt_retry_011
736,evt_retry_005
838,evt_retry_009
1054,evt_retry_002
1133,evt_retry_012


user_id не соответствуют формату: 0


,user_id


source_ip не соответствуют формату: 0


...

,частота
department,
SOC,269
R&D,248
HR,196
Operations,179
IT,176
Sales,174
Finance,166
NaN,14
unknown,8



=== auth_method (уникальных: 4, пропусков: 0) ===


,частота
auth_method,
password,571
push,372
otp,330
fido2,157



=== analyst_verdict (уникальных: 2, пропусков: 0) ===


,частота
analyst_verdict,
benign,1301
confirmed_threat,129


### Решения по результатам проверки типов и допустимых значений

#### `event_id`

**Обнаружено:** в датасете есть строки, не соответствующие шаблону `evt_XXXXXX`.

**Решение:** удалить строки с невалидным значением.

**Обоснование:** невалидный идентификатор не позволяет однозначно сопоставить событие с журналами и указывает на ошибку сбора.

#### `timestamp`

**Обнаружено:** 3 строки не парсятся в `datetime`.

**Решение:** удалить строки с неваридным значением.

**Обоснование:** без корректного времени невозможно применить временное разделение.

#### `hour`

**Обнаружено:** 1 расхождение с `timestamp.dt.hour`

**Решение:** если `timestamp` валиден, `hour` восстанавливается из него.

**Обоснование:** время события — источник, `hour` является его производной. Расхождение трактуется как техническая ошибка сбора, а не как независимая информация.

#### Бинарные признаки (`is_weekend`, `known_device`, `new_country`, `vpn_used`, `is_suspicious`)

**Обнаружено:** все значения в `{0, 1}`, пропусков нет.

**Решение:** оставить без изменений.

#### `failed_attempts_24h`

**Обнаружено:** 3 отрицательных значения.

**Решение:** удалить записи с отрицательным значением.

**Обоснование:** отрицательное число неуспешных попыток физически невозможно и указывает на ошибку данных.

#### `distance_from_usual_km`

**Обнаружено:** 4 отрицательных значения.

**Решение:** удалить записи с отрицательным значением.

**Обоснование:** отрицательное расстояние невозможно -> значение указывает на ошибку сбора данных.

#### `account_age_days`

**Обнаружено:** 3 отрицательных значения.

**Решение:** удалить записи с отрицательным значением.

**Обоснование:** возраст учетной записи в днях не может быть отрицательным.

#### `password_age_days`

**Обнаружено:** 9 нечисловых значений (`unknown`).

**Решение:** удалить записи со значением `unknown` и записи, где возраст пароля превышает возраст учетной записи.

**Обоснование:** `unknown` — отсутствие данных о возрасте пароля, чего быть не может. Пароль не может быть старше самой учетной записи, поэтому соотношение `password_age_days > account_age_days` физически невозможно и указывает на ошибку данных.

#### `browser_family`

**Обнаружено:** 5 уникальных значений, среди которых `NaN` (30) и `-` (10), реальные категории — `Chrome`, `Safari`, `Edge`, `Firefox`.

**Решение:** записи с `NaN` и `-` удалить.

**Обоснование:** `NaN` и `-` означают, что семейство браузера неизвестно, строка не несет информации для фокусного признака варианта 3.

#### `department`

**Обнаружено:** 7 реальных категорий, `NaN` — 14, `unknown` — 8.

**Решение:** записи с `NaN` и `unknown` удалить

**Обоснование:** `NaN` и `unknown` означают отсутствие данных о подразделении -> строка не может быть использована для анализа контекста.

In [5]:
# Условные обозначения, эквивалентные пропуску
CONVENTIONAL_NA = {'?', 'unknown', '-', '', 'nan', 'null',
                   'none', 'n/a', 'na', 'not_recorded', 'undefined'}

def count_missing(series):
    n_nan = int(series.isna().sum())

    if series.dtype == object or str(series.dtype) in ('string', 'category'):
        s_norm = series.astype(str).str.strip().str.lower()
        # исключаем уже посчитанные NaN: astype(str) превращает их в 'nan'
        mask_hidden = s_norm.isin(CONVENTIONAL_NA) & series.notna()
    else:
        mask_hidden = pd.Series(False, index=series.index)

    n_hidden = int(mask_hidden.sum())
    return n_nan, n_hidden, n_nan + n_hidden


rows = []
for col in df_raw.columns:
    n_nan, n_hidden, n_total = count_missing(df_raw[col])
    rows.append({
        'столбец': col,
        'NaN': n_nan,
        'условные': n_hidden,
        'всего_пропусков': n_total,
        'доля_%': round(n_total / len(df_raw) * 100, 2),
    })

missing_stats = (
    pd.DataFrame(rows)
    .query('всего_пропусков > 0')
    .sort_values('доля_%', ascending=False)
    .reset_index(drop=True)
)

print(f'Всего строк (исходно): {len(df_raw)}')
print(f'Столбцов с пропусками (явными или скрытыми): {len(missing_stats)}')
display(missing_stats)

Всего строк (исходно): 1430
Столбцов с пропусками (явными или скрытыми): 5


,столбец,NaN,условные,всего_пропусков,доля_%
0,browser_family,30,10,40,2.80
1,distance_from_usual_km,32,0,32,2.24
2,department,14,8,22,1.54
3,password_age_days,0,9,9,0.63
4,timestamp,0,1,1,0.07


### Дополнительные решения

Выше были описаны действия, предпринимаемые для значений, которые не соответствуют ожидаемому типу.
Среди них не было описано только решение, принятое для `distance_from_usual_km`, значения которых — NaN.
Строки с такими значения должны быть удалены, так как по IP-адресу для всех строк были определены коды стран, а значит и примерное расстояние. То, что значения в ячейке нет означает ошибку в обработке данных, что не подходит для обучения модели.

In [6]:
# Реализация принятых решений по полученным результатам

cleanup_log = []
dropped_ids = []

def drop_and_log(df, mask, rule):
    """Удаляет строки по маске, логирует результат, возвращает новый df."""
    n_before = len(df)
    df = df.loc[~mask].copy()
    n_dropped = n_before - len(df)
    
    cleanup_log.append({
        'правило': rule,
        'удалено_строк': n_dropped,
        'осталось_строк': len(df),
    })
    return df


print(f'Строк до очистки: {len(df)}\n')

# Восстановление hour из timestamp
df.loc[mismatch_hour.index[mismatch_hour], 'hour'] = (df.loc[mismatch_hour.index[mismatch_hour], 'timestamp'].dt.hour)
print(f'hour восстановлен из timestamp для {n_hour_mismatch} строк')

# event_id
mask = ~df['event_id'].map(is_valid_event_id)
df = drop_and_log(
    df, mask,
    rule='event_id не соответствует evt_XXXXXX',
)

# timestamp
mask = df['timestamp'].isna()
df = drop_and_log(
    df, mask,
    rule='timestamp не парсится в datetime',
)

# failed_attempts_24h
vals = pd.to_numeric(df['failed_attempts_24h'], errors='coerce')
mask = (vals < 0) & vals.notna()
df = drop_and_log(
    df, mask,
    rule='failed_attempts_24h < 0',
)

# distance_from_usual_km < 0
vals = pd.to_numeric(df['distance_from_usual_km'], errors='coerce')
mask = (vals < 0) & vals.notna()
df = drop_and_log(
    df, mask,
    rule='distance_from_usual_km < 0',
)

# distance_from_usual_km = NaN
vals = pd.to_numeric(df['distance_from_usual_km'], errors='coerce')
mask = vals.isna()
df = drop_and_log(
    df, mask,
    rule='distance_from_usual_km = NaN',
)

# account_age_days < 0
vals = pd.to_numeric(df['account_age_days'], errors='coerce')
mask = (vals < 0) & vals.notna()
df = drop_and_log(
    df, mask,
    rule='account_age_days < 0',
)

# password_age_days = NaN (бывший unknown)
df['password_age_days'] = pd.to_numeric(df['password_age_days'], errors='coerce')
mask = df['password_age_days'].isna()
df = drop_and_log(
    df, mask,
    rule='password_age_days = unknown',
)

# password_age_days > account_age_days
valid = df[['password_age_days', 'account_age_days']].notna().all(axis=1)
mask = valid & (df['password_age_days'] > df['account_age_days'])
df = drop_and_log(
    df, mask,
    rule='password_age_days > account_age_days',
)

# browser_family: NaN или '-'
mask = df['browser_family'].isna() | (df['browser_family'].astype(str).str.strip() == '-')
df = drop_and_log(
    df, mask,
    rule='browser_family = NaN или "-"',
)

# department: NaN или 'unknown'
mask = df['department'].isna() | (df['department'].astype(str).str.strip().str.lower() == 'unknown')
df = drop_and_log(
    df, mask,
    rule='department = NaN или "unknown"',
)

df = df.reset_index(drop=True)

cleanup_report = pd.DataFrame(cleanup_log)
total_dropped = original_rows - len(df)

print(f'\n=== Итог очистки ===')

display(cleanup_report)

print(f'Было строк:  {original_rows}')
print(f'Стало строк: {len(df)}')
print(f'Удалено:     {total_dropped} ({round(total_dropped / original_rows * 100, 2)} %)')

Строк до очистки: 1430

hour восстановлен из timestamp для 1 строк

=== Итог очистки ===


,правило,удалено_строк,осталось_строк
0,event_id не соответствует evt_XXXXXX,12,1418
1,timestamp не парсится в datetime,3,1415
2,failed_attempts_24h < 0,3,1412
3,distance_from_usual_km < 0,4,1408
4,distance_from_usual_km = NaN,32,1376
5,account_age_days < 0,3,1373
6,password_age_days = unknown,9,1364
7,password_age_days > account_age_days,85,1279
8,"browser_family = NaN или ""-""",37,1242
9,"department = NaN или ""unknown""",21,1221


Было строк:  1430
Стало строк: 1221
Удалено:     209 (14.62 %)


In [7]:
# Преобразование типов к pandas

def convert_series(s, expected_type):
    t = str(expected_type).strip().lower()

    if t == 'category': return s.astype('category')
    if t == 'binary': return s.astype('int8')
    if t == 'integer': return s.astype('int32')
    if t == 'float': return pd.to_numeric(s).astype('float32')
    if t == 'datetime': return pd.to_datetime(s)
    if t == 'string': return s.astype('string')

dtypes_before = df.dtypes.astype(str).copy()

convert_log = []
for col in df.columns:
    expected = expected_fields.get(col)
    if expected is None:
        continue
    dtype_before = type_report['ожидаемый_тип'].to_dict().get(col)
    df[col] = convert_series(df[col], expected)
    dtype_after = str(df[col].dtype)
    convert_log.append({
        'столбец': col,
        'ожидаемый_тип': expected,
        'dtype_до': dtype_before,
        'dtype_после': dtype_after,
    })

display(pd.DataFrame(convert_log))

,столбец,ожидаемый_тип,dtype_до,dtype_после
0,event_id,string,string,string
1,timestamp,datetime,datetime,datetime64[ns]
2,user_id,category,category,category
3,department,category,category,category
4,source_ip,string,string,string
5,country,category,category,category
6,device_type,category,category,category
7,os_family,category,category,category
8,browser_family,category,category,category
9,auth_method,category,category,category
